# Try SendSoon in Google Colab

Calls the same HTTP APIs as [`sendsoon/mcp`](https://github.com/sendsoon/mcp). Uses **`requests` only** (preinstalled in Colab — no `pip install`).

| MCP tool | Endpoint |
| --- | --- |
| `ip_lookup` | `GET /api/ip/lookup` |
| `markitdown_convert` | `POST /api/markitdown/convert` |
| `send_email` | `POST /api/send-test-email` |

Optional env (same as MCP): `SENDSOON_API_KEY`.

The markitdown test uses [`docs/SendSoon_Overview.docx`](https://github.com/sendsoon/mcp/blob/main/docs/SendSoon_Overview.docx) from this repo (downloaded in Colab; read locally if you clone the repo).

In [ ]:
import html
import json
import os
import uuid
from getpass import getpass

import requests

BASE = "https://www.sendsoonai.com"
API_KEY = os.environ.get("SENDSOON_API_KEY", "").strip() or None
recipient = input("Recipient for send_email (blank = skip): ").strip()
key_in = getpass("API Key (Enter = skip): ").strip()
if key_in:
    API_KEY = key_in

def headers(accept="application/json"):
    h = {"Accept": accept}
    if API_KEY:
        h["Authorization"] = f"Bearer {API_KEY}"
    return h

print("Base URL:", BASE)
print("API Key:", "set" if API_KEY else "anonymous trial")
print("Recipient:", recipient or "(skip send_email)")
print()

In [ ]:
# ip_lookup
r = requests.get(f"{BASE}/api/ip/lookup", params={"ip": "8.8.8.8"}, headers=headers(), timeout=30)
r.raise_for_status()
print("ip_lookup:", json.dumps(r.json(), indent=2, ensure_ascii=False))

In [ ]:
# markitdown_convert — uses docs/SendSoon_Overview.docx (local clone or GitHub raw)
from pathlib import Path

SAMPLE_URL = "https://raw.githubusercontent.com/sendsoon/mcp/main/docs/SendSoon_Overview.docx"
SAMPLE_NAME = "SendSoon_Overview.docx"

def load_sample_docx() -> bytes:
    for path in (Path("docs/SendSoon_Overview.docx"), Path("SendSoon_Overview.docx")):
        if path.is_file():
            print(f"Using local: {path.resolve()}")
            return path.read_bytes()
    print(f"Downloading: {SAMPLE_URL}")
    resp = requests.get(SAMPLE_URL, timeout=30)
    resp.raise_for_status()
    return resp.content

docx_bytes = load_sample_docx()
r = requests.post(
    f"{BASE}/api/markitdown/convert",
    headers=headers(accept="text/markdown, application/json"),
    files={"file": (SAMPLE_NAME, docx_bytes)},
    timeout=60,
)
r.raise_for_status()
if "application/json" in (r.headers.get("content-type") or ""):
    print("markitdown:", r.json().get("markdown", ""))
else:
    print("markitdown:", r.text)

In [ ]:
# send_email (skipped when recipient is blank)
if not recipient:
    print("send_email: skipped (no recipient)")
else:
    body = "Configuration successful. Sent from Google Colab."
    r = requests.post(
        f"{BASE}/api/send-test-email",
        headers={**headers(), "Content-Type": "application/json", "Idempotency-Key": str(uuid.uuid4())},
        json={
            "to": recipient,
            "subject": "SendSoon Colab test",
            "htmlContent": f'<pre style="white-space:pre-wrap">{html.escape(body)}</pre>',
        },
        timeout=30,
    )
    print(f"send_email: HTTP {r.status_code}")
    print(r.text)